In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
shell_path = ""
dbox_path = fr'C:\Users\eunic\Dropbox\sa_fires'
root = dbox_path
int_path = f"{root}/proj_bureaucrats_farms/data_output/intermediate"

In [3]:
spam_data = fr"{root}/data/input/crop-production-2010-mapspam"
data = gpd.read_file(f'{spam_data}/spam2010v2r0_global_phys_area.dbf/spam2010V2r0_global_A_TA.DBF')
india = data[data.ISO3 == "IND"]
cols = ['ISO3', 'PROD_LEVEL', 'ALLOC_KEY', 'CELL5M', 'X', 'Y', 'REC_TYPE',
       'TECH_TYPE', 'UNIT','RICE_A', 'CREA_DATE', 'YEAR_DATA', 'SOURCE', 'NAME_CNTR',
       'NAME_ADM1', 'NAME_ADM2']
riceindia = india[cols]

gdf = gpd.GeoDataFrame(
    riceindia, 
    geometry=gpd.points_from_xy(riceindia['X'], riceindia['Y']),
    crs="EPSG:4326"   # change this if your coords are in a different CRS
)

# 2. (Optional) drop the original X/Y if you don’t need them
riceindia_gdf = gdf.drop(columns=['X','Y'])


In [4]:
data = gpd.read_file(f'{spam_data}/spam2010v2r0_global_prod.dbf/spam2010V2r0_global_P_TA.DBF')
india = data[data.ISO3 == "IND"]
cols = ['ISO3', 'PROD_LEVEL', 'ALLOC_KEY', 'CELL5M', 'X', 'Y', 'REC_TYPE',
       'TECH_TYPE', 'UNIT','RICE_A', 'CREA_DATE', 'YEAR_DATA', 'SOURCE', 'NAME_CNTR',
       'NAME_ADM1', 'NAME_ADM2']
riceindiaprod = india[cols]
gdf = gpd.GeoDataFrame(
    riceindiaprod, 
    geometry=gpd.points_from_xy(riceindiaprod['X'], riceindiaprod['Y']),
    crs="EPSG:4326"   # change this if your coords are in a different CRS
)

# 2. (Optional) drop the original X/Y if you don’t need them
riceindiaprod_gdf = gdf.drop(columns=['X','Y'])


In [5]:
data = gpd.read_file(f'{spam_data}/spam2010v2r0_global_harv_area.dbf/spam2010V2r0_global_H_TA.DBF')
india = data[data.ISO3 == "IND"]
cols = ['ISO3', 'PROD_LEVEL', 'ALLOC_KEY', 'CELL5M', 'X', 'Y', 'REC_TYPE',
       'TECH_TYPE', 'UNIT','RICE_A', 'CREA_DATE', 'YEAR_DATA', 'SOURCE', 'NAME_CNTR',
       'NAME_ADM1', 'NAME_ADM2']
riceindiaharvarea = india[cols]
gdf = gpd.GeoDataFrame(
    riceindiaharvarea, 
    geometry=gpd.points_from_xy(riceindiaharvarea['X'], riceindiaharvarea['Y']),
    crs="EPSG:4326"   # change this if your coords are in a different CRS
)

# 2. (Optional) drop the original X/Y if you don’t need them
riceindiaharvarea_gdf = gdf.drop(columns=['X','Y'])


In [6]:
# 1. Create a GeoDataFrame by “injecting” a geometry column
gdf = gpd.GeoDataFrame(
    riceindia, 
    geometry=gpd.points_from_xy(riceindia['X'], riceindia['Y']),
    crs="EPSG:4326"   # change this if your coords are in a different CRS
)

# 2. (Optional) drop the original X/Y if you don’t need them
gdf = gdf.drop(columns=['X','Y'])


In [7]:
acs = gpd.read_file(f"{int_path}/_0_2_3_ACs_right_shapefile.shp")

In [8]:
acs_rice = gpd.overlay(gdf, acs,keep_geom_type=True)

acs_ricearea = acs_rice.groupby(['ac_uq_id'], as_index = False)['RICE_A'].sum()

acs_ricearea.columns = ['ac_uq_id', 'rice_area_ha']

In [9]:
acs_riceindiaharvarea = gpd.overlay(riceindiaharvarea_gdf, acs,keep_geom_type=True)

acs_riceharvarea = acs_riceindiaharvarea.groupby(['ac_uq_id'], as_index = False)['RICE_A'].sum()

acs_riceharvarea.columns = ['ac_uq_id', 'rice_harvarea_ha']

In [10]:
acs_riceindiaprod = gpd.overlay(riceindiaprod_gdf, acs,keep_geom_type=True)

acs_acs_riceprod = acs_riceindiaprod.groupby(['ac_uq_id'], as_index = False)['RICE_A'].sum()

acs_acs_riceprod.columns = ['ac_uq_id', 'rice_prod_mt']

In [11]:
acs['areakm2'] = acs.to_crs(7755).area/(1000*1000)

In [12]:
final_df = acs.merge(acs_ricearea, on = 'ac_uq_id', how = 'left') \
    .merge(acs_riceharvarea, on = 'ac_uq_id', how = 'left') \
    .merge(acs_acs_riceprod, on = 'ac_uq_id', how = 'left').drop(['geometry'], axis = 1)
cols = ['rice_area_ha', 'rice_harvarea_ha', 'rice_prod_mt']
newcols = ['rice_area_aclvl_ahigh', 'rice_harvarea_aclvl_ahigh', 'rice_prod_aclvl_ahigh']
for col, newcol in zip(cols, newcols):
    final_df[f"{col}_share"] = final_df[f"{col}"] / acs['areakm2']
    median_val = final_df[f"{col}_share"].median(skipna=True)  # skip NaNs if present
    final_df[f"{newcol}"] = (final_df[f"{col}_share"] > median_val).astype(int)

In [22]:
final_df.to_stata(f"{int_path}/9_rice_info_ac_lvl.dta")

In [27]:
final_df['rice_area_aclvl_ahigh'].isna().sum()

np.int64(0)

In [23]:
final_df.columns

Index(['ac_uq_id', 'STATE_UT', 'DISTRICT', 'ASSEMBLY_1', 'ASSEMBLY',
       'acpost08ID', 'areakm2', 'rice_area_ha', 'rice_harvarea_ha',
       'rice_prod_mt', 'rice_area_ha_share', 'rice_area_aclvl_ahigh',
       'rice_harvarea_ha_share', 'rice_harvarea_aclvl_ahigh',
       'rice_prod_mt_share', 'rice_prod_aclvl_ahigh'],
      dtype='str')

In [20]:
final_df.to_csv(f"{int_path}/9_rice_info_ac_lvl.csv", index = False)

In [21]:
from pathlib import Path
import duckdb

output_dir = Path(int_path)
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "9_rice_info_ac_lvl.csv"
parquet_path = output_dir / "9_rice_info_ac_lvl.parquet"
duckdb_path = output_dir / "9_rice_info_ac_lvl.duckdb"

# Save as CSV
final_df.to_csv(csv_path, index=False)

# Save as Parquet
final_df.to_parquet(
    parquet_path,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

# Save as DuckDB
with duckdb.connect(str(duckdb_path)) as con:
    con.register("final_df_view", final_df)

    con.execute("""
        CREATE OR REPLACE TABLE rice_info_ac_lvl AS
        SELECT *
        FROM final_df_view
    """)

    con.unregister("final_df_view")

print(f"CSV saved to:     {csv_path}")
print(f"Parquet saved to: {parquet_path}")
print(f"DuckDB saved to:  {duckdb_path}")

CSV saved to:     C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\9_rice_info_ac_lvl.csv
Parquet saved to: C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\9_rice_info_ac_lvl.parquet
DuckDB saved to:  C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms\data_output\intermediate\9_rice_info_ac_lvl.duckdb
